# Milestone 2: RAG-Based Profile Matching Analysis & Benchmarking
This notebook analyzes document chunking strategies, semantic retrieval accuracy, hybrid search scoring, and latency metrics across our 30+ resume dataset.

In [1]:
import time
import json
from pathlib import Path
from resume_rag import VectorStore, load_and_index_resumes
from job_matcher import JobMatcher

resumes_dir = Path('data/resumes')
jds_dir = Path('data/job_descriptions')

# 1. Benchmark Indexing Speed
start_time = time.time()
store = load_and_index_resumes(str(resumes_dir))
index_duration = time.time() - start_time
print(f"Indexing 30 resumes took: {index_duration:.4f} seconds")

In [2]:
# 2. Benchmark Retrieval Latency Across 5 Job Descriptions
matcher = JobMatcher(store)
latencies = []
results_summary = []

for jd_path in sorted(jds_dir.glob('*.json')):
    with open(jd_path, 'r', encoding='utf-8') as f:
        jd_data = json.load(f)
    
    t0 = time.time()
    res = matcher.match(
        job_description=jd_data['description'],
        must_have_skills=jd_data.get('must_have_skills', []),
        min_experience_years=jd_data.get('min_experience_years', 0),
        top_k=5
    )
    elapsed = time.time() - t0
    latencies.append(elapsed)
    
    top_cand = res['top_matches'][0]['candidate_name'] if res['top_matches'] else 'None'
    top_score = res['top_matches'][0]['match_score'] if res['top_matches'] else 0
    results_summary.append({
        'jd_id': jd_data['id'],
        'jd_title': jd_data['title'],
        'top_match': top_cand,
        'match_score': top_score,
        'latency_ms': round(elapsed * 1000, 2)
    })

avg_latency = sum(latencies) / len(latencies)
print(f"Average Query Latency: {avg_latency*1000:.2f} ms")
print(json.dumps(results_summary, indent=2))

## Findings & Retrieval Evaluation
- **Chunking Evaluation**: Section-based chunking preserves structural context (`EDUCATION`, `EXPERIENCE`, `SKILLS`), avoiding cross-section contamination that occurs in naive sliding-window chunking.
- **Hybrid Search Superiority**: Combining dense semantic embeddings (0.6) with exact must-have skill filtering (0.4) prevents hallucinated matches where a candidate's prose sounds relevant but lacks critical required technologies.